# 04. ENTER_NOW probability TCN

모든 인접 positive를 학습하는 기존 ZONE hazard 모델과 분리해, positive episode마다 한
행뿐인 위험·시간 최적 `target_enter_now`를 학습한다.

- class weight와 oversampling을 사용하지 않는다.
- 날짜 expanding OOF와 내부 날짜 epoch 선택 후 과거 전체 refit을 사용한다.
- ENTER_NOW 확률과 기존 ZONE 확률을 OOF에서 결합한다.
- 마지막 날짜는 threshold 선택에 사용하지 않는다.

In [1]:
from pathlib import Path
import gc
import json
import math
import random
import time

import numpy as np
import pandas as pd
import torch
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'AGENT.md').is_file() and (candidate / 'README.md').is_file():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')


SEED = 77
BASE_VERSION = 'scalp_30m_ohlcv_net3_multihorizon_5m_v2'
TARGET_VERSION = 'scalp_30m_ohlcv_zone_entry_5m_v3'
ZONE_MODEL_VERSION = 'buy_multihorizon_hazard_tcn_v1'
MODEL_VERSION = 'enter_now_probability_tcn_v1'
MAX_EPOCHS = 18
PATIENCE = 4
BATCH_SIZE = 1024
LEARNING_RATE = 4e-4
WEIGHT_DECAY = 1e-3
STAKE_USD = 100.0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.set_float32_matmul_precision('high')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PROJECT_ROOT = find_project_root()
PREPROCESS_ROOT = (PROJECT_ROOT / '../../results/preprocessing').resolve()
RESULT_ROOT = (PROJECT_ROOT / '../../results/modeling').resolve()
MODEL_ROOT = (PROJECT_ROOT / '../../model').resolve()
base_schema = json.loads((PREPROCESS_ROOT / f'{BASE_VERSION}_schema.json').read_text())
target_schema = json.loads((PREPROCESS_ROOT / f'{TARGET_VERSION}_schema.json').read_text())
metadata = pd.read_parquet(PREPROCESS_ROOT / f'{TARGET_VERSION}_metadata.parquet')
with np.load(PREPROCESS_ROOT / f'{BASE_VERSION}_features.npz') as loaded:
    all_sequence = loaded['sequence'].astype(np.float32)
with np.load(PREPROCESS_ROOT / f'{TARGET_VERSION}_targets.npz') as loaded:
    enter_target = loaded['enter_now_target'].astype(np.float32)

default_indices = [base_schema['feature_names'].index(name) for name in base_schema['default_feature_names']]
sequence = np.ascontiguousarray(all_sequence[:, :, default_indices], dtype=np.float32)
del all_sequence
sessions = sorted(metadata['session'].unique())
TEST_SESSION = sessions[-1]
OOF_SESSIONS = sessions[-6:-1]
assert len(sequence) == len(metadata) == len(enter_target)
assert np.array_equal(metadata['feature_index'].to_numpy(), np.arange(len(metadata)))
assert np.array_equal(enter_target, metadata['target_enter_now'].to_numpy())
print('device:', DEVICE, '| sequence:', sequence.shape)
print('entry prevalence:', enter_target.mean(), '| anchors:', int(enter_target.sum()))
print('OOF:', OOF_SESSIONS, '| Test:', TEST_SESSION)

device: cuda | sequence: (72181, 30, 38)
entry prevalence: 0.01562738 | anchors: 1128
OOF: ['session_2026-07-10', 'session_2026-07-13', 'session_2026-07-14', 'session_2026-07-15', 'session_2026-07-16'] | Test: session_2026-07-17


In [2]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def fit_scaler(indices, max_windows=20_000, seed=SEED):
    indices = np.asarray(indices, dtype=np.int64)
    rng = np.random.default_rng(seed)
    if len(indices) > max_windows:
        indices = rng.choice(indices, max_windows, replace=False)
    sample = sequence[indices].reshape(-1, sequence.shape[-1])
    center = np.median(sample, axis=0).astype(np.float32)
    q25, q75 = np.percentile(sample, [25, 75], axis=0)
    scale = np.maximum((q75 - q25).astype(np.float32), 1e-4)
    return center, scale


def scale_rows(indices, center, scale):
    values = (sequence[indices] - center[None, None, :]) / scale[None, None, :]
    return np.clip(values, -10, 10).astype(np.float32)


class ResidualTemporalBlock(nn.Module):
    def __init__(self, channels, dilation, dropout):
        super().__init__()
        self.norm = nn.GroupNorm(1, channels)
        self.depthwise = nn.Conv1d(channels, channels, 3, padding=dilation, dilation=dilation, groups=channels)
        self.pointwise = nn.Sequential(
            nn.Conv1d(channels, channels * 2, 1), nn.GELU(), nn.Dropout(dropout),
            nn.Conv1d(channels * 2, channels, 1),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return x + self.dropout(self.pointwise(self.depthwise(self.norm(x))))


class EnterNowTCN(nn.Module):
    def __init__(self, input_dim, initial_probability, channels=48, dropout=0.15):
        super().__init__()
        self.input_projection = nn.Conv1d(input_dim, channels, 1)
        self.blocks = nn.ModuleList([ResidualTemporalBlock(channels, d, dropout) for d in [1, 2, 4, 8]])
        self.final_norm = nn.GroupNorm(1, channels)
        self.shared = nn.Sequential(
            nn.LayerNorm(channels * 3), nn.Linear(channels * 3, 64), nn.GELU(), nn.Dropout(dropout),
        )
        self.entry_head = nn.Linear(64, 1)
        probability = float(np.clip(initial_probability, 1e-4, 1 - 1e-4))
        with torch.no_grad():
            self.entry_head.bias.fill_(math.log(probability / (1 - probability)))

    def forward(self, x):
        x = self.input_projection(x.transpose(1, 2))
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)
        pooled = torch.cat([x[:, :, -1], x.mean(2), x.amax(2)], dim=1)
        return self.entry_head(self.shared(pooled)).squeeze(1)


def ece(actual, probability, bins=10):
    edges = np.linspace(0, 1, bins + 1); value = 0.0
    for left, right in zip(edges[:-1], edges[1:]):
        mask = (probability >= left) & (probability < right if right < 1 else probability <= right)
        if mask.any(): value += mask.mean() * abs(actual[mask].mean() - probability[mask].mean())
    return float(value)


def metrics(actual, probability):
    probability = np.clip(np.asarray(probability), 1e-7, 1 - 1e-7)
    actual = np.asarray(actual)
    return {
        'samples': len(actual), 'positives': int(actual.sum()), 'prevalence': float(actual.mean()),
        'mean_probability': float(probability.mean()),
        'pr_auc': float(average_precision_score(actual, probability)),
        'roc_auc': float(roc_auc_score(actual, probability)),
        'brier': float(brier_score_loss(actual, probability)),
        'log_loss': float(log_loss(actual, probability, labels=[0, 1])), 'ece': ece(actual, probability),
    }


probe = EnterNowTCN(sequence.shape[-1], enter_target.mean())
print('parameters:', sum(p.numel() for p in probe.parameters()))
del probe

parameters: 50193


In [3]:
def make_loader(indices, center, scale, shuffle, seed):
    dataset = TensorDataset(
        torch.from_numpy(scale_rows(indices, center, scale)),
        torch.from_numpy(enter_target[indices]), torch.as_tensor(indices, dtype=torch.long),
    )
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle,
                      generator=torch.Generator().manual_seed(seed),
                      pin_memory=DEVICE.type == 'cuda', num_workers=0)


def train_epoch(model, loader, optimizer):
    model.train(); total = 0.0; count = 0
    for x, y, _ in loader:
        x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits=model(x); loss=nn.functional.binary_cross_entropy_with_logits(logits,y)
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
        total += loss.item()*len(x); count += len(x)
    return total/max(count,1)


@torch.no_grad()
def predict(model, indices, center, scale):
    loader=make_loader(indices,center,scale,False,SEED); model.eval(); logits=[]; rows=[]
    for x,_,index in loader:
        logits.append(model(x.to(DEVICE,non_blocking=True)).cpu().numpy()); rows.append(index.numpy())
    return np.concatenate(logits),np.concatenate(rows)


def choose_epoch(train_indices,valid_indices,seed):
    set_seed(seed); center,scale=fit_scaler(train_indices,seed=seed)
    loader=make_loader(train_indices,center,scale,True,seed)
    model=EnterNowTCN(sequence.shape[-1],enter_target[train_indices].mean()).to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=MAX_EPOCHS,eta_min=5e-5)
    best,best_epoch,stale=np.inf,1,0; history=[]
    for epoch in range(1,MAX_EPOCHS+1):
        train_loss=train_epoch(model,loader,optimizer)
        logits,_=predict(model,valid_indices,center,scale)
        valid_loss=log_loss(enter_target[valid_indices],expit(logits),labels=[0,1])
        history.append({'epoch':epoch,'train_loss':train_loss,'valid_log_loss':valid_loss})
        if valid_loss < best-1e-5: best,best_epoch,stale=valid_loss,epoch,0
        else: stale+=1
        scheduler.step()
        if stale>=PATIENCE: break
    del model,loader; gc.collect(); torch.cuda.empty_cache() if DEVICE.type=='cuda' else None
    return best_epoch,pd.DataFrame(history)


def fit_fixed(train_indices,epochs,seed):
    set_seed(seed); center,scale=fit_scaler(train_indices,seed=seed)
    loader=make_loader(train_indices,center,scale,True,seed)
    model=EnterNowTCN(sequence.shape[-1],enter_target[train_indices].mean()).to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=max(epochs,1),eta_min=5e-5)
    history=[]
    for epoch in range(1,epochs+1):
        loss=train_epoch(model,loader,optimizer); history.append({'epoch':epoch,'train_loss':loss}); scheduler.step()
    del loader; gc.collect()
    return model,center,scale,pd.DataFrame(history)

## Expanding OOF ENTER_NOW probability

In [4]:
oof_frames=[]; fold_rows=[]; histories=[]; best_epochs=[]; started=time.time()
columns=['sample_id','feature_index','session','symbol','decision_bar_timestamp','entry_timestamp','entry_ask',
         'target_zone_5m','target_enter_first','target_enter_now','first_hit_minute','timeout_net_return_5m',
         'positive_episode_id','positive_episode_position','minutes_from_entry_anchor','entry_quality']
for fold,valid_session in enumerate(OOF_SESSIONS,start=1):
    prior=[s for s in sessions if s<valid_session]; inner_valid=prior[-1]; inner_train=prior[:-1]
    inner_train_idx=np.flatnonzero(metadata.session.isin(inner_train).to_numpy())
    inner_valid_idx=np.flatnonzero(metadata.session.eq(inner_valid).to_numpy())
    refit_idx=np.flatnonzero(metadata.session.isin(prior).to_numpy())
    valid_idx=np.flatnonzero(metadata.session.eq(valid_session).to_numpy())
    epoch,h=choose_epoch(inner_train_idx,inner_valid_idx,SEED+fold); best_epochs.append(epoch)
    h['fold']=fold; h['stage']='inner'; histories.append(h)
    model,center,scale,rh=fit_fixed(refit_idx,epoch,SEED+100+fold)
    rh['fold']=fold; rh['stage']='refit'; histories.append(rh)
    logits,indices=predict(model,valid_idx,center,scale); assert np.array_equal(indices,valid_idx)
    frame=metadata.iloc[valid_idx][columns].copy().reset_index(drop=True)
    frame['raw_entry_logit']=logits; frame['raw_entry_probability']=expit(logits); frame['fold']=fold
    oof_frames.append(frame); m=metrics(enter_target[valid_idx],frame.raw_entry_probability)
    fold_rows.append({'fold':fold,'validation_session':valid_session,'best_epoch':epoch,**m})
    print(f"fold {fold} {valid_session} epoch={epoch} AP={m['pr_auc']:.4f} P={m['mean_probability']:.3%}/{m['prevalence']:.3%}")
    del model; gc.collect(); torch.cuda.empty_cache() if DEVICE.type=='cuda' else None
oof=pd.concat(oof_frames,ignore_index=True).sort_values('entry_timestamp').reset_index(drop=True)
fold_metrics=pd.DataFrame(fold_rows); training_history=pd.concat(histories,ignore_index=True)
print(f'elapsed {(time.time()-started)/60:.2f} min'); display(fold_metrics)

fold 1 session_2026-07-10 epoch=10 AP=0.0729 P=2.061%/2.141%


fold 2 session_2026-07-13 epoch=10 AP=0.0678 P=1.784%/1.848%


fold 3 session_2026-07-14 epoch=5 AP=0.0561 P=1.820%/1.406%


fold 4 session_2026-07-15 epoch=5 AP=0.0551 P=1.811%/1.695%


fold 5 session_2026-07-16 epoch=7 AP=0.0305 P=1.480%/1.369%
elapsed 0.88 min


,fold,validation_session,best_epoch,samples,positives,prevalence,mean_probability,pr_auc,roc_auc,brier,log_loss,ece
0,1,session_2026-07-10,10,10230,219,0.021408,0.020610,0.072942,0.781217,0.020400,0.091661,0.000797
1,2,session_2026-07-13,10,7953,147,0.018484,0.017844,0.067820,0.800967,0.017697,0.080924,0.000639
2,3,session_2026-07-14,5,7043,99,0.014057,0.018198,0.056054,0.828009,0.013555,0.064208,0.004142
3,4,session_2026-07-15,5,9556,162,0.016953,0.018111,0.055085,0.786887,0.016324,0.076554,0.001159
4,5,session_2026-07-16,7,5918,81,0.013687,0.014805,0.030477,0.737547,0.013451,0.068100,0.001118


## Rolling calibration과 raw/calibrated 선택

In [5]:
def fit_platt(frame):
    if frame.target_enter_now.nunique()<2: return {'coef':1.0,'intercept':0.0,'identity':True}
    estimator=LogisticRegression(C=10.0,solver='lbfgs',max_iter=1000)
    estimator.fit(frame[['raw_entry_logit']],frame.target_enter_now)
    return {'coef':float(estimator.coef_[0,0]),'intercept':float(estimator.intercept_[0]),'identity':False}


rolling=[]
for position,session in enumerate(OOF_SESSIONS):
    current=oof[oof.session.eq(session)].copy(); prior=oof[oof.session.isin(OOF_SESSIONS[:position])]
    calibration=fit_platt(prior) if len(prior) else {'coef':1.0,'intercept':0.0,'identity':True}
    current['calibrated_entry_probability']=expit(calibration['coef']*current.raw_entry_logit+calibration['intercept'])
    current['calibration_history_sessions']=position; rolling.append(current)
oof=pd.concat(rolling,ignore_index=True).sort_values('entry_timestamp').reset_index(drop=True)
eligible=oof[oof.calibration_history_sessions.gt(0)]
raw_metrics=metrics(eligible.target_enter_now,eligible.raw_entry_probability)
cal_metrics=metrics(eligible.target_enter_now,eligible.calibrated_entry_probability)
SELECTED_PROBABILITY_TYPE='raw' if (raw_metrics['brier'],raw_metrics['log_loss']) <= (cal_metrics['brier'],cal_metrics['log_loss']) else 'calibrated'
ENTRY_PROBABILITY_COLUMN='raw_entry_probability' if SELECTED_PROBABILITY_TYPE=='raw' else 'calibrated_entry_probability'
FINAL_CALIBRATION=fit_platt(oof)
print('raw:',raw_metrics); print('calibrated:',cal_metrics); print('selected:',SELECTED_PROBABILITY_TYPE)

raw: {'samples': 30470, 'positives': 489, 'prevalence': 0.016048572366261896, 'mean_probability': 0.01741948164999485, 'pr_auc': 0.053261345221508605, 'roc_auc': 0.7911141268815853, 'brier': 0.015484566800296307, 'log_loss': 0.0731992315440674, 'ece': 0.0013709087229221027}
calibrated: {'samples': 30470, 'positives': 489, 'prevalence': 0.016048572366261896, 'mean_probability': 0.017429739236831665, 'pr_auc': 0.053421806780711345, 'roc_auc': 0.7904172301626067, 'brier': 0.015485540963709354, 'log_loss': 0.07325600858320776, 'ece': 0.0013811677762786787}
selected: raw


## 기존 ZONE OOF와 결합 threshold

In [6]:
zone_oof=pd.read_parquet(RESULT_ROOT/f'{ZONE_MODEL_VERSION}_oof_predictions.parquet',columns=['feature_index','raw_tp_by_probability_5m'])
combined_oof=oof.merge(zone_oof,on='feature_index',how='inner',validate='one_to_one')
threshold_frame=combined_oof.copy()

def sequential_trades(frame,signal_column):
    candidates=frame[frame[signal_column]].sort_values('entry_timestamp').copy(); selected=[]
    for _,group in candidates.groupby(['session','symbol'],sort=False):
        available=None
        for index,row in group.sort_values('entry_timestamp').iterrows():
            if available is not None and row.entry_timestamp<available: continue
            selected.append(index); hold=int(row.first_hit_minute) if row.target_zone_5m else 5
            available=row.entry_timestamp+pd.Timedelta(minutes=hold)
    trades=candidates.loc[selected].copy()
    trades['policy_return']=np.where(trades.target_zone_5m.eq(1),.03,trades.timeout_net_return_5m)
    return trades


def policy_metrics(frame,signal_column):
    trades=sequential_trades(frame,signal_column)
    if len(trades)==0: return {'trades':0,'sessions_traded':0,'zone_precision':np.nan,'entry_precision':np.nan,'entry_recall':0.0,'mean_abs_timing_error':np.nan,'mean_return':np.nan,'risk_adjusted_return':np.nan,'total_pnl_usd':0.0,'profit_factor':np.nan}
    r=trades.policy_return.to_numpy(); pnl=r*STAKE_USD; gp=pnl[pnl>0].sum(); gl=-pnl[pnl<0].sum()
    timed=trades[trades.target_zone_5m.eq(1)&trades.minutes_from_entry_anchor.notna()]
    return {'trades':len(trades),'sessions_traded':trades.session.nunique(),'zone_precision':trades.target_zone_5m.mean(),
            'entry_precision':trades.target_enter_now.mean(),'entry_recall':trades.target_enter_now.sum()/max(frame.target_enter_now.sum(),1),
            'mean_abs_timing_error':timed.minutes_from_entry_anchor.abs().mean() if len(timed) else np.nan,
            'mean_return':r.mean(),'risk_adjusted_return':r.mean()-r.std(ddof=1)/math.sqrt(len(r)) if len(r)>1 else r.mean(),
            'total_pnl_usd':pnl.sum(),'profit_factor':gp/gl if gl>0 else np.inf}


zone_thresholds=np.unique(np.round(np.r_[np.arange(.01,.151,.02),threshold_frame.raw_tp_by_probability_5m.quantile([.5,.7,.8,.9,.95,.975])],6))
entry_thresholds=np.unique(np.round(np.r_[np.arange(.0025,.0501,.0025),[.075,.1,.15,.2],threshold_frame[ENTRY_PROBABILITY_COLUMN].quantile([.5,.7,.8,.9,.95,.975,.99])],6))
rows=[]
for z in zone_thresholds:
    for e in entry_thresholds:
        threshold_frame['candidate_signal']=threshold_frame.raw_tp_by_probability_5m.ge(z)&threshold_frame[ENTRY_PROBABILITY_COLUMN].ge(e)
        rows.append({'zone_threshold':z,'entry_threshold':e,**policy_metrics(threshold_frame,'candidate_signal')})
table=pd.DataFrame(rows); candidates=table[table.trades.ge(50)&table.sessions_traded.ge(3)]
selected=candidates.sort_values(['risk_adjusted_return','total_pnl_usd']).iloc[-1]
ZONE_THRESHOLD=float(selected.zone_threshold); ENTRY_THRESHOLD=float(selected.entry_threshold)
OOF_ELIGIBLE=bool(selected.risk_adjusted_return>0 and selected.profit_factor>1)
threshold_frame['selected_signal']=threshold_frame.raw_tp_by_probability_5m.ge(ZONE_THRESHOLD)&threshold_frame[ENTRY_PROBABILITY_COLUMN].ge(ENTRY_THRESHOLD)
oof_trades=sequential_trades(threshold_frame,'selected_signal')
print('selected',ZONE_THRESHOLD,ENTRY_THRESHOLD,'eligible',OOF_ELIGIBLE); display(selected.to_frame('value')); display(table.sort_values('risk_adjusted_return',ascending=False).head(20))

selected 0.01 0.0025 eligible False


,value
zone_threshold,0.010000
entry_threshold,0.002500
trades,5915.000000
sessions_traded,5.000000
zone_precision,0.065427
entry_precision,0.024007
entry_recall,0.200565
mean_abs_timing_error,1.165375
mean_return,-0.010752
risk_adjusted_return,-0.011163


,zone_threshold,entry_threshold,trades,sessions_traded,zone_precision,entry_precision,entry_recall,mean_abs_timing_error,mean_return,risk_adjusted_return,total_pnl_usd,profit_factor
432,0.155574,0.150000,2,2,0.500000,0.500000,0.001412,0.000000,0.052527,0.030000,10.505412,inf
401,0.150000,0.150000,2,2,0.500000,0.500000,0.001412,0.000000,0.052527,0.030000,10.505412,inf
215,0.081572,0.150000,2,2,0.500000,0.500000,0.001412,0.000000,0.052527,0.030000,10.505412,inf
184,0.070000,0.150000,2,2,0.500000,0.500000,0.001412,0.000000,0.052527,0.030000,10.505412,inf
370,0.137360,0.150000,2,2,0.500000,0.500000,0.001412,0.000000,0.052527,0.030000,10.505412,inf
339,0.130000,0.150000,2,2,0.500000,0.500000,0.001412,0.000000,0.052527,0.030000,10.505412,inf
308,0.113163,0.150000,2,2,0.500000,0.500000,0.001412,0.000000,0.052527,0.030000,10.505412,inf
29,0.010000,0.150000,2,2,0.500000,0.500000,0.001412,0.000000,0.052527,0.030000,10.505412,inf
153,0.057917,0.150000,2,2,0.500000,0.500000,0.001412,0.000000,0.052527,0.030000,10.505412,inf
60,0.024095,0.150000,2,2,0.500000,0.500000,0.001412,0.000000,0.052527,0.030000,10.505412,inf


## Final Test와 저장

In [7]:
FINAL_EPOCHS=int(np.rint(np.median(best_epochs)))
train_idx=np.flatnonzero(metadata.session.ne(TEST_SESSION).to_numpy()); test_idx=np.flatnonzero(metadata.session.eq(TEST_SESSION).to_numpy())
model,center,scale,final_history=fit_fixed(train_idx,FINAL_EPOCHS,SEED+999)
test_logits,test_rows=predict(model,test_idx,center,scale); assert np.array_equal(test_rows,test_idx)
test=metadata.iloc[test_idx][columns].copy().reset_index(drop=True)
test['raw_entry_logit']=test_logits; test['raw_entry_probability']=expit(test_logits)
test['calibrated_entry_probability']=expit(FINAL_CALIBRATION['coef']*test_logits+FINAL_CALIBRATION['intercept'])
TEST_ENTRY_COLUMN='raw_entry_probability' if SELECTED_PROBABILITY_TYPE=='raw' else 'calibrated_entry_probability'
zone_test=pd.read_parquet(RESULT_ROOT/f'{ZONE_MODEL_VERSION}_test_predictions.parquet',columns=['feature_index','raw_tp_by_probability_5m'])
test=test.merge(zone_test,on='feature_index',how='inner',validate='one_to_one')
test['buy_signal']=test.raw_tp_by_probability_5m.ge(ZONE_THRESHOLD)&test[TEST_ENTRY_COLUMN].ge(ENTRY_THRESHOLD)
test_policy=policy_metrics(test,'buy_signal'); test_trades=sequential_trades(test,'buy_signal')
entry_test_metrics=metrics(test.target_enter_now,test[TEST_ENTRY_COLUMN])
DEPLOYMENT_ELIGIBLE=bool(OOF_ELIGIBLE and test_policy['trades']>=20 and test_policy['risk_adjusted_return']>0 and test_policy['profit_factor']>1)

model_path=MODEL_ROOT/f'{MODEL_VERSION}.pt'; metrics_path=RESULT_ROOT/f'{MODEL_VERSION}_metrics.json'
torch.save({'model_version':MODEL_VERSION,'architecture':'EnterNowTCN','state_dict':{k:v.detach().cpu() for k,v in model.state_dict().items()},
            'feature_names':base_schema['default_feature_names'],'scaler_center':torch.from_numpy(center),'scaler_scale':torch.from_numpy(scale),
            'selected_probability_type':SELECTED_PROBABILITY_TYPE,'calibration':FINAL_CALIBRATION,
            'research_zone_threshold':ZONE_THRESHOLD,'research_entry_threshold':ENTRY_THRESHOLD,
            'authorized_thresholds':{'zone':ZONE_THRESHOLD,'entry':ENTRY_THRESHOLD} if DEPLOYMENT_ELIGIBLE else None,
            'deployment_eligible':DEPLOYMENT_ELIGIBLE},model_path)
fold_metrics.to_parquet(RESULT_ROOT/f'{MODEL_VERSION}_fold_metrics.parquet',index=False,compression='zstd')
training_history.to_parquet(RESULT_ROOT/f'{MODEL_VERSION}_training_history.parquet',index=False,compression='zstd')
oof.to_parquet(RESULT_ROOT/f'{MODEL_VERSION}_oof_predictions.parquet',index=False,compression='zstd')
test.to_parquet(RESULT_ROOT/f'{MODEL_VERSION}_test_predictions.parquet',index=False,compression='zstd')
table.to_parquet(RESULT_ROOT/f'{MODEL_VERSION}_threshold_candidates.parquet',index=False,compression='zstd')
test_trades.to_parquet(RESULT_ROOT/f'{MODEL_VERSION}_test_trades.parquet',index=False,compression='zstd')
payload={'model_version':MODEL_VERSION,'parameters':sum(p.numel() for p in model.parameters()),'best_epochs':best_epochs,'final_epochs':FINAL_EPOCHS,
         'selected_probability_type':SELECTED_PROBABILITY_TYPE,'oof_probability_metrics':raw_metrics if SELECTED_PROBABILITY_TYPE=='raw' else cal_metrics,
         'test_probability_metrics':entry_test_metrics,'zone_threshold':ZONE_THRESHOLD,'entry_threshold':ENTRY_THRESHOLD,
         'selected_oof_policy':{k:(None if pd.isna(v) else float(v)) for k,v in selected.to_dict().items()},
         'test_policy':{k:(None if pd.isna(v) else float(v)) for k,v in test_policy.items()},
         'oof_economic_eligible':OOF_ELIGIBLE,'deployment_eligible':DEPLOYMENT_ELIGIBLE}
metrics_path.write_text(json.dumps(payload,ensure_ascii=False,indent=2),encoding='utf-8')
print('Test entry metrics',entry_test_metrics); print('Test policy',test_policy); print('deployment',DEPLOYMENT_ELIGIBLE)
display(test_trades.head(30))

Test entry metrics {'samples': 6609, 'positives': 56, 'prevalence': 0.008473293993039794, 'mean_probability': 0.01057352777570486, 'pr_auc': 0.03336378646271867, 'roc_auc': 0.8183983344596805, 'brier': 0.00832127034664154, 'log_loss': 0.04328460385573885, 'ece': 0.0021002348860476895}
Test policy {'trades': 681, 'sessions_traded': 1, 'zone_precision': np.float64(0.039647577092511016), 'entry_precision': np.float64(0.01908957415565345), 'entry_recall': np.float64(0.23214285714285715), 'mean_abs_timing_error': np.float64(1.1851851851851851), 'mean_return': np.float64(-0.010204703908034315), 'risk_adjusted_return': np.float64(-0.011276366258173794), 'total_pnl_usd': np.float64(-694.9403361371369), 'profit_factor': np.float64(0.3375218397274125)}
deployment False


,sample_id,feature_index,session,symbol,decision_bar_timestamp,entry_timestamp,entry_ask,target_zone_5m,target_enter_first,target_enter_now,...,positive_episode_id,positive_episode_position,minutes_from_entry_anchor,entry_quality,raw_entry_logit,raw_entry_probability,calibrated_entry_probability,raw_tp_by_probability_5m,buy_signal,policy_return
547,scalp_30m_ohlcv_net3_multihorizon_5m_v2_00066119,66119,session_2026-07-17,BIYA,2026-07-17 08:49:00+00:00,2026-07-17 08:50:00+00:00,3.93,0,0,0,...,<NA>,0,NaN,NaN,-5.042025,0.006419,0.006515,0.016670,True,0.002482
552,scalp_30m_ohlcv_net3_multihorizon_5m_v2_00066124,66124,session_2026-07-17,BIYA,2026-07-17 08:54:00+00:00,2026-07-17 08:55:00+00:00,3.98,0,0,0,...,<NA>,0,NaN,NaN,-5.053661,0.006345,0.006443,0.017491,True,-0.025187
557,scalp_30m_ohlcv_net3_multihorizon_5m_v2_00066129,66129,session_2026-07-17,BIYA,2026-07-17 08:59:00+00:00,2026-07-17 09:00:00+00:00,3.89,0,0,0,...,<NA>,0,NaN,NaN,-4.543466,0.010525,0.010465,0.029594,True,-0.010346
563,scalp_30m_ohlcv_net3_multihorizon_5m_v2_00066135,66135,session_2026-07-17,BIYA,2026-07-17 09:05:00+00:00,2026-07-17 09:06:00+00:00,3.88,0,0,0,...,<NA>,0,NaN,NaN,-5.193475,0.005522,0.005640,0.012819,True,-0.002641
568,scalp_30m_ohlcv_net3_multihorizon_5m_v2_00066140,66140,session_2026-07-17,BIYA,2026-07-17 09:10:00+00:00,2026-07-17 09:11:00+00:00,3.91,0,0,0,...,<NA>,0,NaN,NaN,-5.423914,0.004390,0.004527,0.013965,True,-0.005178
573,scalp_30m_ohlcv_net3_multihorizon_5m_v2_00066145,66145,session_2026-07-17,BIYA,2026-07-17 09:15:00+00:00,2026-07-17 09:16:00+00:00,3.91,0,0,0,...,<NA>,0,NaN,NaN,-5.651085,0.003501,0.003644,0.012259,True,-0.017965
585,scalp_30m_ohlcv_net3_multihorizon_5m_v2_00066157,66157,session_2026-07-17,BIYA,2026-07-17 09:27:00+00:00,2026-07-17 09:28:00+00:00,3.94,0,0,0,...,<NA>,0,NaN,NaN,-5.744699,0.003189,0.003333,0.011436,True,0.010089
590,scalp_30m_ohlcv_net3_multihorizon_5m_v2_00066162,66162,session_2026-07-17,BIYA,2026-07-17 09:32:00+00:00,2026-07-17 09:33:00+00:00,3.99,0,0,0,...,<NA>,0,NaN,NaN,-5.141742,0.005814,0.005925,0.012281,True,-0.015099
613,scalp_30m_ohlcv_net3_multihorizon_5m_v2_00066185,66185,session_2026-07-17,BIYA,2026-07-17 09:55:00+00:00,2026-07-17 09:56:00+00:00,3.99,0,0,0,...,<NA>,0,NaN,NaN,-4.536088,0.010602,0.010538,0.016277,True,-0.020112
618,scalp_30m_ohlcv_net3_multihorizon_5m_v2_00066190,66190,session_2026-07-17,BIYA,2026-07-17 10:00:00+00:00,2026-07-17 10:01:00+00:00,3.94,0,0,0,...,<NA>,0,NaN,NaN,-3.939322,0.019090,0.018522,0.014601,True,0.005013
